# Sample of GFandSlope usage on Google Colab. environment

## Import google.colab file module

In [1]:
from google.colab import files

## Check CUDA environment

In [2]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!nvidia-smi --query-gpu=compute_cap --format=csv,noheader
!nvcc --version

NVIDIA L4
8.9
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## Get code and sample data from repository

In [3]:
!git clone https://github.com/AiGIS-PyAiGIS/GFandSlope/

Cloning into 'GFandSlope'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 54 (delta 14), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 1.56 MiB | 4.58 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [4]:
!ls GFandSlope/

input_sample  README.md  src  util


In [5]:
%cd GFandSlope/src

/content/GFandSlope/src


## Build code

In [6]:
!make clean; make

gcc -Wno-unused-result -O   -c  -lm GFandSlope.c
gcc -Wno-unused-result -O   -c  -lm GF_host.c
Detected GPU name: NVIDIA L4
Detected GPU compute capability: sm_89
/usr/local/cuda/bin/nvcc GF.cu -O4  -lineinfo -c --ptxas-options=-v -arch=sm_89
ptxas info    : 0 bytes gmem, 136 bytes cmem[3]
ptxas info    : Compiling entry function '_Z18CalcForAllPolygonsPK4Vec3PK5PLISTPK9Edge_listP6OutputPS_' for 'sm_89'
ptxas info    : Function properties for _Z18CalcForAllPolygonsPK4Vec3PK5PLISTPK9Edge_listP6OutputPS_
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 96 registers, used 0 barriers, 392 bytes cmem[0], 368 bytes cmem[2]
ptxas info    : Compile time = 207.419 ms
gcc -Wno-unused-result -O   -DSP -c  -lm model_IO.c
/usr/local/cuda/bin/nvcc -o GFandSlope GFandSlope.o GF_host.o GF.o model_IO.o  -lm 
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-t

In [7]:
!cp GFandSlope ../
%cd ../

/content/GFandSlope


In [8]:
!cp input_sample/* .

## Example: compute without a point list (with the centers of face)

In [9]:
!cat input_Itokawa_ro1950_P12.1324h_49k.txt

// PERIOD[hour], DENSITY[kg/m^3]
period: 12.1324
density: 1950
input_polygon: Itokawa_49152.txt
input_points: NONE
output: Itokawa_49k_ro1950_P12.1324h.txt
gpu: 0


In [10]:
!./GFandSlope input_Itokawa_ro1950_P12.1324h_49k.txt

   --- General Information for device 0 ---
Name:  NVIDIA L4
Compute capability:  8.9
Clock rate:  2040000
Device copy overlap:  Enabled
Kernel execution timeout :  Disabled
   --- Memory Information for device 0 ---
Total global mem:  23659151360
Total constant Mem:  65536
Max mem pitch:  2147483647
Texture Alignment:  512
   --- MP Information for device 0 ---
Multiprocessor count:  58
Shared mem per mp:  49152
Registers per mp:  65536
Threads in warp:  32
Max threads per block:  1024
Max thread dimensions:  (1024, 1024, 64)
Max grid dimensions:  (2147483647, 65535, 65535)

=================== Calculation Settings ===================
Loaded from input_Itokawa_ro1950_P12.1324h_49k.txt

PERIOD: 12.132400 [hour]
DENSITY: 1950.000000 [kg/m^3]
Input Model File: Itokawa_49152.txt
No Points File: Polygon centers will be used.
Output File: Itokawa_49k_ro1950_P12.1324h.txt
Target Device(GPU) ID: 0

Loading Model Data...
>>> Vertices
>>> Triplet of Polygons
>>> a
>>> b
>>> c
Done.
Points Data 

In [11]:
!head Itokawa_49k_ro1950_P12.1324h.txt

# Shape model: Itokawa_49152.txt
# Number of Polygons: 49152
# Point list: NONE
# Number of Points: 49152 (polygon centers)
# Density: 1950.000000 [kg/m^3]
# Rotational Period: 12.132400 [h]
# Unit: All coordinates in km, and SI units for other values.
ID Point.x Point.y Point.z Lon Lat CRefAcc.x CRefAcc.y CRefAcc.z GravAcc.x GravAcc.y GravAcc.z TotalAcc.x TotalAcc.y TotalAcc.z Gpotential Rpotential Tpotential GeopotentialSlope Normal.x Normal.y Normal.z Area 
1 -0.148537 0.081457 0.076603 151.259886 24.331928 -3.073937e-06 1.685730e-06 -0.000000e+00 -3.268631e-05 3.469742e-05 6.217096e-05 -2.961237e-05 3.301169e-05 6.217096e-05 -1.367096e-02 2.969531e-04 -1.396791e-02 6.260537e+00 -4.514500e+00 5.024000e+00 1.112460e+01 6.507265e-06 
2 -0.149840 0.080393 0.076493 151.785203 24.220273 -3.100909e-06 1.663724e-06 -0.000000e+00 -3.322878e-05 3.395309e-05 6.226257e-05 -3.012787e-05 3.228936e-05 6.226257e-05 -1.366910e-02 2.991963e-04 -1.396830e-02 7.982436e+00 -4.861900e+00 4.164300e+00 1.

In [12]:
files.download("Itokawa_49k_ro1950_P12.1324h.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Example: compute with a point list

In [13]:
!cat input_Itokawa_ro1950_P12.1324h_49k_center.txt

// PERIOD[hour], DENSITY[kg/m^3]
period: 12.1324
density: 1950
input_polygon: Itokawa_49152.txt
input_points: Itokawa_49152_center.txt
output: Itokawa_49k_ro1950_P12.1324h_center.txt
gpu: 0


In [14]:
!./GFandSlope input_Itokawa_ro1950_P12.1324h_49k_center.txt

   --- General Information for device 0 ---
Name:  NVIDIA L4
Compute capability:  8.9
Clock rate:  2040000
Device copy overlap:  Enabled
Kernel execution timeout :  Disabled
   --- Memory Information for device 0 ---
Total global mem:  23659151360
Total constant Mem:  65536
Max mem pitch:  2147483647
Texture Alignment:  512
   --- MP Information for device 0 ---
Multiprocessor count:  58
Shared mem per mp:  49152
Registers per mp:  65536
Threads in warp:  32
Max threads per block:  1024
Max thread dimensions:  (1024, 1024, 64)
Max grid dimensions:  (2147483647, 65535, 65535)

=================== Calculation Settings ===================
Loaded from input_Itokawa_ro1950_P12.1324h_49k_center.txt

PERIOD: 12.132400 [hour]
DENSITY: 1950.000000 [kg/m^3]
Input Model File: Itokawa_49152.txt
Input Points File: Itokawa_49152_center.txt
Output File: Itokawa_49k_ro1950_P12.1324h_center.txt
Target Device(GPU) ID: 0

Loading Model Data...
>>> Vertices
>>> Triplet of Polygons
>>> a
>>> b
>>> c
Done.


In [15]:
!head Itokawa_49k_ro1950_P12.1324h_center.txt

# Shape model: Itokawa_49152.txt
# Number of Polygons: 49152
# Point list: Itokawa_49152_center.txt
# Number of Points: 49152
# Density: 1950.000000 [kg/m^3]
# Rotational Period: 12.132400 [h]
# Unit: All coordinates in km, and SI units for other values.
ID Point.x Point.y Point.z Lon Lat CRefAcc.x CRefAcc.y CRefAcc.z GravAcc.x GravAcc.y GravAcc.z TotalAcc.x TotalAcc.y TotalAcc.z Gpotential Rpotential Tpotential 
1 -0.148537 0.081457 0.076603 151.259886 24.331928 -3.073937e-06 1.685730e-06 -0.000000e+00 -3.268631e-05 3.469742e-05 6.217096e-05 -2.961237e-05 3.301169e-05 6.217096e-05 -1.367096e-02 2.969531e-04 -1.396791e-02 
2 -0.149840 0.080393 0.076493 151.785203 24.220273 -3.100909e-06 1.663724e-06 -0.000000e+00 -3.322878e-05 3.395309e-05 6.226257e-05 -3.012787e-05 3.228936e-05 6.226257e-05 -1.366910e-02 2.991963e-04 -1.396830e-02 


In [16]:
files.download("Itokawa_49k_ro1950_P12.1324h_center.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Example: utilities for OBJ file and an on-the-fly input file

In [17]:
!rm *.txt

In [18]:
!pip install pyvista -qq &> /dev/null

In [19]:
!python util/Mesh2GFandSlopeInput.py Itokawa_49152.obj

In [20]:
!python util/GetFaceCentersOfMesh.py Itokawa_49152.obj

In [21]:
!ls

GFandSlope    Itokawa_49152.obj		    Itokawa_49152.obj.txt  src
input_sample  Itokawa_49152.obj.center.txt  README.md		   util


In [22]:
!head Itokawa_49152.obj.txt

25350
1 -0.15153 0.08183 0.07523
2 -0.14726 0.08251 0.07657
3 -0.14288 0.08309 0.07759
4 -0.13851 0.08366 0.07864
5 -0.13436 0.08447 0.08038
6 -0.13023 0.08532 0.082
7 -0.12588 0.08588 0.08284
8 -0.12135 0.08618 0.08325
9 -0.11685 0.08652 0.08376


In [23]:
!head Itokawa_49152.obj.center.txt

49152
-1.485166666666666857e-01 8.144000000000001238e-02 7.656666666666667176e-02
-1.498366666666666736e-01 8.038666666666666183e-02 7.649666666666667114e-02
-1.480933333333333268e-01 7.888333333333333308e-02 7.757666666666666877e-02
-1.494266666666666521e-01 7.780666666666667675e-02 7.728000000000000147e-02
-1.476900000000000157e-01 7.629666666666666541e-02 7.834666666666667556e-02
-1.490566666666666706e-01 7.525333333333333874e-02 7.818333333333334079e-02
-1.472800000000000220e-01 7.369333333333333291e-02 7.913333333333333330e-02
-1.485933333333333550e-01 7.260666666666666658e-02 7.883333333333333859e-02
-1.468000000000000138e-01 7.107666666666666300e-02 7.969999999999999307e-02


In [24]:
setupfile = 'input.txt'
shapemodel = 'Itokawa_49152.obj.txt'
output = 'Itokawa_49k_ro1950_P12.1324h_center_2.txt'
points = 'Itokawa_49152.obj.center.txt'
density = '1950'
period = '12.1324'
with open(setupfile, mode='w') as f:
    f.write('// PERIOD[hour], DENSITY[kg/m^3]\n')
    f.write('period: ' + period + '\n')
    f.write('density: ' + density + '\n')
    f.write('input_polygon: ' + shapemodel + '\n')
    f.write('input_points: ' + points + '\n')
    f.write('output: ' + output + '\n')

In [25]:
!cat input.txt

// PERIOD[hour], DENSITY[kg/m^3]
period: 12.1324
density: 1950
input_polygon: Itokawa_49152.obj.txt
input_points: Itokawa_49152.obj.center.txt
output: Itokawa_49k_ro1950_P12.1324h_center_2.txt


In [26]:
!./GFandSlope input.txt

   --- General Information for device 0 ---
Name:  NVIDIA L4
Compute capability:  8.9
Clock rate:  2040000
Device copy overlap:  Enabled
Kernel execution timeout :  Disabled
   --- Memory Information for device 0 ---
Total global mem:  23659151360
Total constant Mem:  65536
Max mem pitch:  2147483647
Texture Alignment:  512
   --- MP Information for device 0 ---
Multiprocessor count:  58
Shared mem per mp:  49152
Registers per mp:  65536
Threads in warp:  32
Max threads per block:  1024
Max thread dimensions:  (1024, 1024, 64)
Max grid dimensions:  (2147483647, 65535, 65535)

=================== Calculation Settings ===================
Loaded from input.txt

PERIOD: 12.132400 [hour]
DENSITY: 1950.000000 [kg/m^3]
Input Model File: Itokawa_49152.obj.txt
Input Points File: Itokawa_49152.obj.center.txt
Output File: Itokawa_49k_ro1950_P12.1324h_center_2.txt
Target Device(GPU) ID: 0

Loading Model Data...
>>> Vertices
>>> Triplet of Polygons
>>> a
>>> b
>>> c
Done.
Loading Points Data...
>>>

In [27]:
files.download(output)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>